# Two-Stage BEC Detection Pipeline
This notebook splits the email into `header` (Subject, From, Reply-To) and `body`. It passes them to two separate machine learning models (Header Machine and Body Machine) and merges the outputs for final classification.

In [5]:
import pandas as pd
import os
import re

path = r"..\..\Dataset"
dataframes = {}

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            df_name = os.path.splitext(file)[0]
            file_path = os.path.join(root, file)
            
            if os.path.getsize(file_path) == 0:
                continue

            try:
                current_df = pd.read_csv(file_path)
                
                # Specifically handle "emails.csv" (often the Enron dataset of legitimate emails)
                if df_name.lower() == 'emails' and not current_df.empty and 'label' not in current_df.columns:
                    if 'message' in current_df.columns:
                        current_df['label'] = 0  # Assuming all are Legitimate (0)
                    if 'file' in current_df.columns:
                        current_df = current_df.drop(columns=['file'])
                
                if not current_df.empty and 'label' in current_df.columns:
                    dataframes[df_name] = current_df
                    print(f"Loaded {df_name}")
            except Exception as e:
                pass

Loaded CEAS_08
Loaded email_phishing_data
Loaded Enron
Loaded Ling
Loaded Nazario
Loaded Nigerian_Fraud
Loaded phishing_email
Loaded SpamAssasin


### 1. Separate Header and Body during Data Standardization

In [6]:
cleaned_dfs = []

# Heuristic to separate header from body
def extract_header_body(text):
    if not isinstance(text, str):
        return "", ""
    # Standard raw emails use \n\n to separate headers and body
    parts = re.split(r'\n\s*\n', text, maxsplit=1)
    if len(parts) == 2:
        return parts[0], parts[1]
    
    # Fallback heuristic if no double newline is found
    # Assume first 150 chars contains Subject/From if not structurally split
    return text[:150], text[150:]

for name, d in dataframes.items():
    temp_df = d.copy()
    text_cols = [col for col in temp_df.columns if col != 'label']
    
    temp_df[text_cols] = temp_df[text_cols].fillna("").astype(str)
    temp_df['text_combined'] = temp_df[text_cols].agg(' '.join, axis=1)
    
    # Extract header and body
    temp_df[['header', 'body']] = temp_df['text_combined'].apply(lambda x: pd.Series(extract_header_body(x)))
    
    temp_df['label'] = pd.to_numeric(temp_df['label'], errors='coerce')
    temp_df = temp_df.dropna(subset=['label']).copy()
    temp_df['label'] = temp_df['label'].astype(int)
    
    cleaned_dfs.append(temp_df[['header', 'body', 'label']])

df = pd.concat(cleaned_dfs, ignore_index=True)

print("Merged Data Shape Before Cleaning:", df.shape)
display(df.head())

Merged Data Shape Before Cleaning: (689818, 3)


,header,body,label
0,Young Esposito <Young@iworld.de> user4@gvc.cea...,come. Even as Nazi tanks were rolling down the...,1
1,Mok <ipline's1983@icable.ph> user2.2@gvc.ceas-...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1
3,Michael Parker <ivqrnai@pobox.com> SpamAssassi...,For more info on the .so domain you can read a...,0
4,Gretchen Suggs <externalsep1@loanofficertool.c...,1,1


### 1.1 Data Cleaning - Remove Duplicates

In [7]:
print("--- Remove Duplicates & Missing Values ---")

rows_before = len(df)
duplicate_rows_before = df.duplicated().sum()

print(f"Rows before: {rows_before}")
print(f"Duplicate rows found: {duplicate_rows_before}")

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Remove NA/missing values
na_rows_before = df.isna().any(axis=1).sum()
print(f"Missing (NA) rows found: {na_rows_before}")
df = df.dropna().copy()

rows_after = len(df)
removed_total = rows_before - rows_after

print(f"Rows after cleaning: {rows_after}")
print(f"Total rows removed (Duplicates + NAs): {removed_total}")

--- Remove Duplicates & Missing Values ---
Rows before: 689818
Duplicate rows found: 320216
Missing (NA) rows found: 0
Rows after cleaning: 369602
Total rows removed (Duplicates + NAs): 320216


### 1.1.1 Export Cleaned Dataset to Excel

In [8]:
# Install openpyxl if it isn't already installed, as pandas needs it to write Excel files
%pip install openpyxl

import re

# Excel cannot handle certain hidden control characters often found in raw emails.
# We'll remove those illegal characters before exporting.
illegal_chars = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')
df['header'] = df['header'].str.replace(illegal_chars, '', regex=True)
df['body'] = df['body'].str.replace(illegal_chars, '', regex=True)



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 1.2 Text Preprocessing & Tokenization
Clean and tokenize both the `header` and `body` text.

In [9]:
import nltk
from nltk.tokenize import word_tokenize

# Ensure standard tokenizers are available
nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)

print("--- Tokenizing Header and Body ---")

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    # Tokenize and convert to lowercase
    return word_tokenize(text.lower())

# Apply tokenization
df['header_tokenized'] = df['header'].apply(tokenize_text)
df['body_tokenized'] = df['body'].apply(tokenize_text)

print("Sample of tokenized features:")
display(df[['header_tokenized', 'body_tokenized']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


--- Tokenizing Header and Body ---
Sample of tokenized features:


,header_tokenized,body_tokenized
0,"[young, esposito, <, young, @, iworld.de, >, u...","[come, ., even, as, nazi, tanks, were, rolling..."
1,"[mok, <, ipline's1983, @, icable.ph, >, user2....",[1]
2,"[daily, top, 10, <, karmandeep-opengevl, @, un...",[=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=...
3,"[michael, parker, <, ivqrnai, @, pobox.com, >,...","[for, more, info, on, the, .so, domain, you, c..."
4,"[gretchen, suggs, <, externalsep1, @, loanoffi...",[1]


### 2. Feature Extraction (TF-IDF & Word2Vec representation)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Note: The vectorizers below use the raw string ('header', 'body'), 
# which TF-IDF handles smoothly, but we've successfully stored tokenized forms if needed for other embeddings.

# TF-IDF for Headers (Machine 1)
header_vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
X_train_header = header_vectorizer.fit_transform(train_df['header'])
X_test_header = header_vectorizer.transform(test_df['header'])

# TF-IDF for Body (Machine 2) 
# Note: You can replace this with Word2Vec embeddings as per your diagram
body_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_body = body_vectorizer.fit_transform(train_df['body'])
X_test_body = body_vectorizer.transform(test_df['body'])

y_train = train_df['label'].values
y_test = test_df['label'].values


### 3. Machine 1: Header Classifier Comparison
Compare multiple classification models (Random Forest, XGBoost, Naive Bayes) strictly on the email headers.

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Reduced complexity for faster training
header_models = {
    'Random Forest': RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=30, max_depth=5, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

header_results = []
header_train_preds = {}
header_test_preds = {}
best_header_acc = 0
best_header_model_name = ""

print("--- Training Header Machines ---")
for name, model in header_models.items():
    print(f"Training {name} on headers...")
    model.fit(X_train_header, y_train)
    
    y_pred = model.predict(X_test_header)
    acc = accuracy_score(y_test, y_pred)
    header_results.append({'Model': name, 'Header Accuracy': acc})
    
    # Store predictions for stacking later
    header_train_preds[name] = model.predict_proba(X_train_header)[:, 1]
    header_test_preds[name] = model.predict_proba(X_test_header)[:, 1]
    
    if acc > best_header_acc:
        best_header_acc = acc
        best_header_model_name = name

header_df = pd.DataFrame(header_results).sort_values(by='Header Accuracy', ascending=False)
print("\nHeader Models Comparison:")
display(header_df)

print(f"\nBest Header Model: {best_header_model_name}")
# Set the best model's predictions to be used in the final stacking
best_header_train_preds = header_train_preds[best_header_model_name]
best_header_test_preds = header_test_preds[best_header_model_name]

--- Training Header Machines ---
Training Random Forest on headers...
Training XGBoost on headers...
Training Naive Bayes on headers...
Training KNN on headers...

Header Models Comparison:


,Model,Header Accuracy
2,Naive Bayes,0.948878
3,KNN,0.935823
1,XGBoost,0.911919
0,Random Forest,0.828912



Best Header Model: Naive Bayes


### 4. Machine 2: Body Classifier Comparison
Compare multiple classification models (Naive Bayes, XGBoost, Random Forest) strictly on the email bodies.

In [12]:
# Reduced complexity for faster training
body_models = {
    'Random Forest': RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=30, max_depth=5, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1) # KNN might still be slow during prediction
}

body_results = []
body_train_preds = {}
body_test_preds = {}
best_body_acc = 0
best_body_model_name = ""

print("--- Training Body Machines ---")
for name, model in body_models.items():
    print(f"Training {name} on bodies...")
    model.fit(X_train_body, y_train)
    
    y_pred = model.predict(X_test_body)
    acc = accuracy_score(y_test, y_pred)
    body_results.append({'Model': name, 'Body Accuracy': acc})
    
    # Store predictions for stacking later
    body_train_preds[name] = model.predict_proba(X_train_body)[:, 1]
    body_test_preds[name] = model.predict_proba(X_test_body)[:, 1]
    
    if acc > best_body_acc:
        best_body_acc = acc
        best_body_model_name = name

body_df = pd.DataFrame(body_results).sort_values(by='Body Accuracy', ascending=False)
print("\nBody Models Comparison:")
display(body_df)

print(f"\nBest Body Model: {best_body_model_name}")
# Set the best model's predictions to be used in the final stacking
best_body_train_preds = body_train_preds[best_body_model_name]
best_body_test_preds = body_test_preds[best_body_model_name]

--- Training Body Machines ---
Training Random Forest on bodies...
Training XGBoost on bodies...
Training Naive Bayes on bodies...
Training KNN on bodies...

Body Models Comparison:


,Model,Body Accuracy
3,KNN,0.916817
2,Naive Bayes,0.911838
1,XGBoost,0.907049
0,Random Forest,0.824624



Best Body Model: KNN


### 5. Final Classification (Cascade / Stacking)
Combine the outputs of the Header and Body machines.

In [13]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Stack the probability features of the best header model and best body model
X_train_final = np.column_stack((best_header_train_preds, best_body_train_preds))
X_test_final = np.column_stack((best_header_test_preds, best_body_test_preds))

print(f"--- Training Final Classifier (Stacking {best_header_model_name} Header + {best_body_model_name} Body) ---")
final_model = LogisticRegression()
final_model.fit(X_train_final, y_train)

final_preds = final_model.predict(X_test_final)
final_acc = accuracy_score(y_test, final_preds)

print("\nFinal Combined Classification Report:")
print(classification_report(y_test, final_preds))
print(f"Final Cascaded Accuracy: {final_acc:.4f}")

--- Training Final Classifier (Stacking Naive Bayes Header + KNN Body) ---

Final Combined Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     55559
           1       0.97      0.89      0.93     18362

    accuracy                           0.97     73921
   macro avg       0.97      0.94      0.95     73921
weighted avg       0.97      0.97      0.96     73921

Final Cascaded Accuracy: 0.9654


### 5.1 Export Best Header and Body Results
Save the highest-accuracy header and body model outputs into the `outputs/RandomForest Two stage FUlldataset` folder.

In [14]:
from pathlib import Path

output_dir = Path('..') / 'outputs' / 'RandomForest Two stage FUlldataset'
output_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {
        'Stage': 'Header',
        'Best Model': best_header_model_name,
        'Accuracy': best_header_acc
    },
    {
        'Stage': 'Body',
        'Best Model': best_body_model_name,
        'Accuracy': best_body_acc
    },
    {
        'Stage': 'Final Combined',
        'Best Model': f'{best_header_model_name} + {best_body_model_name}',
        'Accuracy': final_acc
    }
])

header_df.to_csv(output_dir / 'header_models.csv', index=False)
body_df.to_csv(output_dir / 'body_models.csv', index=False)
summary_df.to_csv(output_dir / 'summary_metrics.csv', index=False)

print(f"Best Header Model: {best_header_model_name} | Accuracy: {best_header_acc:.4f}")
print(f"Best Body Model: {best_body_model_name} | Accuracy: {best_body_acc:.4f}")
print(f"Saved outputs to: {output_dir}")
display(summary_df)

Best Header Model: Naive Bayes | Accuracy: 0.9489
Best Body Model: KNN | Accuracy: 0.9168
Saved outputs to: ..\outputs\RandomForest Two stage FUlldataset


,Stage,Best Model,Accuracy
0,Header,Naive Bayes,0.948878
1,Body,KNN,0.916817
2,Final Combined,Naive Bayes + KNN,0.965368


### 6. Detailed Evaluation Reports
Full classification reports (precision, recall, f1-score, support) for every header and body model individually.

In [15]:
from sklearn.metrics import classification_report

print("="*50)
print("HEADER MODELS - DETAILED REPORTS")
print("="*50)
for name, model in header_models.items():
    print(f"\n--- Header Model: {name} ---")
    # Get predictions for the test set
    y_pred_header = model.predict(X_test_header)
    # Print the full classification report (precision, recall, f1-score, support, accuracy, macro avg, weighted avg)
    print(classification_report(y_test, y_pred_header))

print("\n" + "="*50)
print("BODY MODELS - DETAILED REPORTS")
print("="*50)
for name, model in body_models.items():
    print(f"\n--- Body Model: {name} ---")
    # Get predictions for the test set
    y_pred_body = model.predict(X_test_body)
    # Print the full classification report
    print(classification_report(y_test, y_pred_body))

HEADER MODELS - DETAILED REPORTS

--- Header Model: Random Forest ---
              precision    recall  f1-score   support

           0       0.82      0.99      0.90     55559
           1       0.94      0.33      0.49     18362

    accuracy                           0.83     73921
   macro avg       0.88      0.66      0.69     73921
weighted avg       0.85      0.83      0.80     73921


--- Header Model: XGBoost ---
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     55559
           1       0.96      0.67      0.79     18362

    accuracy                           0.91     73921
   macro avg       0.93      0.83      0.87     73921
weighted avg       0.92      0.91      0.91     73921


--- Header Model: Naive Bayes ---
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     55559
           1       0.91      0.88      0.90     18362

    accuracy                           0.95   